# Reto de Laboratorio: Preparando la Matriz Tensorial de la Liga ⚽🤖


Contexto de Negocio:
El equipo de analítica quedó fascinado con el Almacén de Datos (OLAP) que construyeron la semana pasada. Ahora, el departamento de Inteligencia Artificial quiere dar el siguiente paso: entrenar una Red Neuronal para predecir cuántos goles anotará un jugador la próxima temporada.El Científico de Datos les acaba de enviar la tabla resultante del cubo OLAP, pero se detuvo por un error de Scikit-Learn: el algoritmo no sabe leer posiciones como "Forward" ni jerarquías como "Oro".Su misión como Ingenieros de Machine Learning es aplicar Feature Engineering para traducir esta tabla humana a una Matriz $\mathbf{X}$ puramente matemática, sin destruir la lógica vectorial.

In [39]:
import pandas as pd
import numpy as np

# 1. Ingesta de datos desde el repositorio crudo
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/players_data-2025_2026.csv"
df_scouting = pd.read_csv(url)

# 2. SIMULACIÓN: El equipo de analistas agregó una columna de jerarquía de talento
# (Evaluación cualitativa del jugador)
np.random.seed(42)
df_scouting['scout_tier'] = np.random.choice(['Bronce', 'Plata', 'Oro', 'Diamante'], size=len(df_scouting))

# Filtramos las columnas que usaremos para el modelo predictivo
columnas_ml = ['Player', 'Squad', 'Age', 'Pos', 'scout_tier', 'Min', 'Gls'] #Esta línea estaba mal, los nombres de columnas están bien raras en el DF
matriz_base = df_scouting[columnas_ml].copy()

print("--- DATOS BASE PARA EL MODELO (Legibles para humanos) ---")
display(matriz_base.head(10))

--- DATOS BASE PARA EL MODELO (Legibles para humanos) ---


,Player,Squad,Age,Pos,scout_tier,Min,Gls
0,Brenden Aaronson,Leeds United,24.0,"MF,FW",Oro,2449,4
1,Jerome Abbey,Wolves,15.0,MF,Diamante,17,0
2,Zach Abbott,Nottingham Forest,19.0,DF,Bronce,164,0
3,Jones El-Abdellaoui,Celta Vigo,19.0,"MF,FW",Oro,681,2
4,Himad Abdelli,Marseille,25.0,MF,Oro,138,0
5,Himad Abdelli,Angers,25.0,MF,Diamante,943,2
6,Ali Abdi,Nice,31.0,"MF,DF",Bronce,1187,3
7,Salis Abdul Samed,Nice,25.0,MF,Bronce,911,0
8,Saud Abdulhamid,Lens,26.0,MF,Oro,1352,2
9,Tay Abed,Levante,21.0,MF,Plata,93,0


### FASE 1: El Traductor Jerárquico (Label Encoding)

Observa la columna `scout_tier`. Contiene valores de texto: 'Bronce', 'Plata', 'Oro', 'Diamante'. Esta variable tiene un **orden natural** claro (Diamante es superior a Bronce).

**Tu tarea:**
Crea un diccionario de mapeo que asigne números enteros respetando la jerarquía natural del talento (iniciando en 1). Luego, aplica este mapeo a la base de datos para crear una nueva columna llamada `scout_encoded`.

In [40]:
scout_tier = {
    'Bronce': 1,
    'Plata': 2,
    'Oro': 3,
    'Diamante': 4
}
matriz_base['scout_encoded'] = matriz_base['scout_tier'].map(scout_tier)

print("Resultado del Label Encoding:")
display(matriz_base[['Player', 'scout_tier', 'scout_encoded']].head())

Resultado del Label Encoding:


,Player,scout_tier,scout_encoded
0,Brenden Aaronson,Oro,3
1,Jerome Abbey,Diamante,4
2,Zach Abbott,Bronce,1
3,Jones El-Abdellaoui,Oro,3
4,Himad Abdelli,Oro,3


### FASE 2: El Traductor Nominal (One-Hot Encoding)

Ahora observa la columna `position`. Contiene valores como 'Forward' (Delantero), 'Midfielder' (Mediocampista), 'Defender' (Defensa), etc.
Esta variable es **Nominal**. Si le asignamos números (1 al Delantero y 3 al Defensa), la Red Neuronal creerá falsamente que un Defensa "vale el triple" que un Delantero.

**Tu tarea:**
Utiliza la función de Pandas correspondiente para destruir la columna `position` y expandirla en múltiples columnas booleanas (0 y 1). Guarda el resultado en una nueva variable llamada `matriz_X`.

In [41]:
"""
Solo una pequeña nota, voy a seguir las instrucciones de la tarea pero me di cuenta que en el DF original, en la columna de posicion vienen 
entradas con multiples valores, como la de brenden Aaronson que tiene MF y FW, lo que al aplicar el One-Hot
genera una columna extra para tomar ambos valores, por lo que voy a aplicar un split para solo tomar el primer valor de la posición
"""

matriz_base['Pos'] = matriz_base['Pos'].str.split(',').str[0]
# CÓDIGO ESQUELETO PARA EL ALUMNO:
matriz_X = pd.get_dummies(matriz_base[['Pos']], columns=['Pos'])

matriz_X = matriz_X.join(matriz_base[['Player', 'scout_tier' , 'scout_encoded',]], how='left')
print("Resultado del One-Hot Encoding:")
display(matriz_X.head(10))

Resultado del One-Hot Encoding:


,Pos_DF,Pos_FW,Pos_GK,Pos_MF,Player,scout_tier,scout_encoded
0,False,False,False,True,Brenden Aaronson,Oro,3
1,False,False,False,True,Jerome Abbey,Diamante,4
2,True,False,False,False,Zach Abbott,Bronce,1
3,False,False,False,True,Jones El-Abdellaoui,Oro,3
4,False,False,False,True,Himad Abdelli,Oro,3
5,False,False,False,True,Himad Abdelli,Diamante,4
6,False,False,False,True,Ali Abdi,Bronce,1
7,False,False,False,True,Salis Abdul Samed,Bronce,1
8,False,False,False,True,Saud Abdulhamid,Oro,3
9,False,False,False,True,Tay Abed,Plata,2


### FASE 3: Purgando la Basura Humana

La Inteligencia Artificial está casi lista para devorar los datos. Sin embargo, si le enviamos la matriz ahora mismo, el algoritmo fallará porque todavía hay texto puro estorbando.

**Tu tarea:**
Elimina de `matriz_X` todas las columnas que ya no le sirven al algoritmo:
1. Las columnas de texto original que ya transformaste (como `scout_tier`).
2. Las columnas descriptivas/operativas que no aportan valor matemático universal (`player_name`, `team_name`).

In [42]:
matriz_X = matriz_X.drop(columns=['Player', 'scout_tier'])

print("--- MATRIZ X DEFINITIVA (Lista para Scikit-Learn / PyTorch) ---")
display(matriz_X.head())
print("\nTipos de datos finales (Todos deben ser numéricos):")
print(matriz_X.dtypes)

--- MATRIZ X DEFINITIVA (Lista para Scikit-Learn / PyTorch) ---


,Pos_DF,Pos_FW,Pos_GK,Pos_MF,scout_encoded
0,False,False,False,True,3
1,False,False,False,True,4
2,True,False,False,False,1
3,False,False,False,True,3
4,False,False,False,True,3



Tipos de datos finales (Todos deben ser numéricos):
Pos_DF            bool
Pos_FW            bool
Pos_GK            bool
Pos_MF            bool
scout_encoded    int64
dtype: object


### 🧠 FASE 4: Defensa Arquitectónica (Responde aquí mismo)

Imagina que el CTO de la empresa revisa tu código y te hace dos preguntas críticas. Responde con argumentos de ingeniería:

**Pregunta 1:** *"Noté que borraste la columna `team_name` en lugar de aplicarle One-Hot Encoding. En nuestra base de datos mundial de la FIFA tenemos 4,000 equipos diferentes. Si le hubieras aplicado One-Hot Encoding a la columna de equipos, ¿qué le habría pasado a la estructura matemática de tu Matriz X y a la memoria RAM de nuestro servidor?"*
> **Tu Respuesta:** Porque al haber al menos 4000 equipos, al aplicar One-Hot encoding generaría 4000 columnas para representar cada equipo individualmente, lo cual sobrecargaría a la memoria RAM y causaría que el modelo sea extremadamente lento y por lo tanto, inútil.

**Pregunta 2:** *"¿Por qué usamos Label Encoding (1, 2, 3, 4) para la evaluación del scout (`scout_tier`), pero no usamos esa misma técnica matemática para la posición en el campo (`position`)?"*
> **Tu Respuesta:** Porque scout_tier si representa una posición jerárquica, por lo hace sentido al aplicar label encoding porque tier uno tiene menos valor que tier 4, tal como funciona en el mundo real hablando con los títulos como 'bronce' o 'diamante'. Por otra parte, cada posición no tiene mayor dominancia jerárquica una contra otra, es solo el papel en el que se desencvuelve un jugador, pero no representa su habilidad técnica, por lo que no hará sentido aplicar la misma lógica de jerarquía que se usa en label encoding.